<a href="https://colab.research.google.com/github/danielpazrosseboe/MasterDRC/blob/main/TFRecords_Band_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import auth
auth.authenticate_user()

!gcloud config set project master-thesis-measles -q


INFORMATION: Project 'master-thesis-measles' has no 'environment' tag set. Use either 'Production', 'Development', 'Test', or 'Staging'. Add an 'environment' tag using `gcloud resource-manager tags bindings create`.
Updated property [core/project].


In [ ]:
# ======================
# DHS-style TFRecord band analysis — CLEAN GCS VERSION
# - Streams from gs://ssadata
# - Safe sequential reading, no watchdog
# - Saves summary CSV + JSON to Drive
# ======================
!pip -q install tqdm tensorflow numpy pandas

import os, re, time, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm

# --- CONFIG ---
TFRECORD_DIR = "gs://ssadata"   # your GCS bucket path
OUT_DIR      = "/content/drive/My Drive/Data Validation/Band Analysis"
os.makedirs(OUT_DIR, exist_ok=True)

BANDS = ['BLUE','GREEN','RED','SWIR1','SWIR2','TEMP1','NIR','NIGHTLIGHTS']
PATCH_SIZE = 255
PIXELS_PER_IMAGE = PATCH_SIZE**2

# --- HELPERS ---
def parse_example(raw):
    ex = tf.train.Example.FromString(raw)
    return ex.features.feature

def get_arrays(f):
    out = {}
    for b in BANDS:
        out[b] = np.asarray(f[b].float_list.value, dtype=np.float32) if b in f else np.zeros(PATCH_SIZE**2, np.float32)
    return out

def update_stats(stats, arrays, good_mask):
    for i, b in enumerate(BANDS):
        arr = arrays[b]
        if arr.size == 0:
            continue
        stats["mins"][i] = min(stats["mins"][i], arr.min())
        stats["maxs"][i] = max(stats["maxs"][i], arr.max())
        stats["sums"][i] += arr.sum(dtype=np.float64)
        stats["sum_sqs"][i] += np.square(arr, dtype=np.float64).sum()
        nz = np.count_nonzero(arr > 0)
        stats["nz_pixels"][i] += nz
        if nz > 0:
            stats["mins_nz"][i] = min(stats["mins_nz"][i], arr[arr > 0].min())
        if good_mask.any():
            stats["mins_goodpx"][i] = min(stats["mins_goodpx"][i], arr[good_mask].min())
    return stats

# --- INIT STATS ---
nbands = len(BANDS)
stats = {
    "mins": np.full(nbands, np.inf),
    "mins_nz": np.full(nbands, np.inf),
    "mins_goodpx": np.full(nbands, np.inf),
    "maxs": np.full(nbands, -np.inf),
    "sums": np.zeros(nbands, np.float64),
    "sum_sqs": np.zeros(nbands, np.float64),
    "nz_pixels": np.zeros(nbands, np.int64),
}
num_good_pixels = []
skipped = []

# --- FILE LIST (GCS) ---
pattern = TFRECORD_DIR.rstrip("/") + "/*.tfrecord.gz"
paths = sorted(tf.io.gfile.glob(pattern))
if not paths:
    raise SystemExit(f"No .tfrecord.gz files found in {TFRECORD_DIR}")
print(f"Found {len(paths)} TFRecords in GCS. Beginning analysis...")

# --- MAIN LOOP ---
start = time.time()
for path in tqdm(paths, desc="Processing TFRecords"):
    try:
        ds = tf.data.TFRecordDataset(path, compression_type="GZIP")
        for raw in ds.as_numpy_iterator():
            f = parse_example(raw)
            arrays = get_arrays(f)
            good_mask = np.zeros(PATCH_SIZE**2, dtype=bool)
            for b in BANDS:
                good_mask |= (arrays[b] > 0)
            stats = update_stats(stats, arrays, good_mask)
            num_good_pixels.append(int(good_mask.sum()))
    except Exception as e:
        skipped.append((path, str(e)))
        continue

elapsed = time.time() - start
images_count = len(num_good_pixels)
total_pixels = images_count * PIXELS_PER_IMAGE
print(f"\nProcessed {images_count} images in {elapsed:.1f}s")
if skipped:
    print(f"Skipped {len(skipped)} files (see skipped.json).")

# --- COMPUTE SUMMARY ---
def compute_summary(stats, num_good_pixels):
    total_good = max(np.sum(num_good_pixels), 1)
    total_pixels = len(num_good_pixels) * PIXELS_PER_IMAGE

    means = stats["sums"]/max(total_pixels,1)
    stds  = np.sqrt(np.maximum(0, stats["sum_sqs"]/max(total_pixels,1) - means**2))

    nz = np.maximum(stats["nz_pixels"], 1)
    means_nz = stats["sums"]/nz
    stds_nz  = np.sqrt(np.maximum(0, stats["sum_sqs"]/nz - means_nz**2))

    means_good = stats["sums"]/total_good
    stds_good  = np.sqrt(np.maximum(0, stats["sum_sqs"]/total_good - means_good**2))

    df = pd.DataFrame({
        "band": BANDS,
        "mean_all": means,
        "std_all": stds,
        "min_all": stats["mins"],
        "max_all": stats["maxs"],
        "mean_nz": means_nz,
        "std_nz": stds_nz,
        "min_nz": stats["mins_nz"],
        "min_good": stats["mins_goodpx"],
        "max_good": stats["maxs"],
        "nz_pixels": stats["nz_pixels"],
    })
    df["avg_good_pixels"] = np.mean(num_good_pixels)
    return df

summary_df = compute_summary(stats, num_good_pixels)

# --- SAVE OUTPUTS ---
csv_path = os.path.join(OUT_DIR, "band_stats_summary.csv")
json_path = os.path.join(OUT_DIR, "band_stats_summary.json")
skipped_path = os.path.join(OUT_DIR, "skipped_files.json")

summary_df.to_csv(csv_path, index=False)
summary_df.to_json(json_path, orient="records", indent=2)
with open(skipped_path, "w") as f:
    json.dump(skipped, f, indent=2)

print(f"\nSaved summary CSV → {csv_path}")
print(f"Saved summary JSON → {json_path}")
print(f"Saved skipped file log → {skipped_path}")


Found 819 TFRecords in GCS. Beginning analysis...


Processing TFRecords:   0%|          | 0/819 [00:00<?, ?it/s]

In [ ]:
# %matplotlib inline
!pip -q install tensorflow matplotlib numpy pandas

import os, re, math, io, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from collections import defaultdict

# --- Paths ---
TFRECORD_DIR = "gs://ssadata"          # folder with *.tfrecord.gz
OUT_DIR      = "/content/drive/My Drive/Data Validation/Band Analysis"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# --- Bands & order ---
BAND_ORDER        = ['BLUE','GREEN','RED','SWIR1','SWIR2','TEMP1','NIR','NIGHTLIGHTS']
BAND_ORDER_NLSPLIT= ['BLUE','GREEN','RED','SWIR1','SWIR2','TEMP1','NIR','DMSP','VIIRS']  # if you used split

# --- Image geometry ---
PATCH = 255
PIXELS_PER_IMAGE = PATCH * PATCH

# --- TEMP1 Kelvin conversion for reporting only ---
ST10_SCALE  = 0.00341802
ST10_OFFSET = 149.0
def temp_to_kelvin(arr):
    a = np.asarray(arr, np.float32)
    if a.size == 0: return a
    return a * ST10_SCALE + ST10_OFFSET if float(np.nanmax(a)) > 5_000 else a

# --- TFRecord feature helpers ---
SCALAR_FLOAT_KEYS = ["lon","lat","wealthpooled"]

def parse_example(raw):
    ex = tf.train.Example.FromString(raw)
    return ex.features.feature

def get_arrays(fmap, bands=BAND_ORDER):
    out = {}
    for b in bands:
        if b in fmap:
            out[b] = np.asarray(fmap[b].float_list.value, dtype=np.float32)
        else:
            out[b] = np.array([], dtype=np.float32)
    return out

def first_example_feature_map(tfrecord_path):
    ds = tf.data.TFRecordDataset(tfrecord_path, compression_type="GZIP")
    for raw in ds:
        return parse_example(raw.numpy())
    return None

def list_tfrecords(pattern_dir):
    return sorted(tf.io.gfile.glob(pattern_dir.rstrip("/") + "/*.tfrecord.gz"))

# Pick one file
files = list_tfrecords(TFRECORD_DIR)
assert files, "No TFRecord files found."
ONE = files[0]
print("Inspecting:", os.path.basename(ONE))

f = first_example_feature_map(ONE)
assert f is not None, "Empty TFRecord."

# Print available keys
print("Feature keys:", sorted(list(f.keys()))[:20], "...")

# Print scalars if present
for k in SCALAR_FLOAT_KEYS:
    if k in f:
        v = f[k].float_list.value
        print(f"{k}:", float(v[0]) if len(v) else None)

# Show quick per-band min/max for the first example
arrs = get_arrays(f, BAND_ORDER)
for b, a in arrs.items():
    if a.size == PIXELS_PER_IMAGE:
        if b == "TEMP1":
            ak = temp_to_kelvin(a)
            print(f"{b}: DN[{a.min():.2f},{a.max():.2f}] → K[{ak.min():.2f},{ak.max():.2f}]")
        else:
            print(f"{b}: min={np.nanmin(a):.4f} max={np.nanmax(a):.4f} mean={np.nanmean(a):.4f}")
    else:
        print(f"{b}: missing or wrong size ({a.size})")

# Plot histograms for the same example
fig, axes = plt.subplots(2, 4, figsize=(14,6))
axes = axes.ravel()

for i, b in enumerate(BAND_ORDER):
    ax = axes[i]
    a = arrs[b]
    if a.size != PIXELS_PER_IMAGE:
        ax.set_title(f"{b} (missing)")
        ax.axis("off"); continue
    if b == "TEMP1":
        vals = temp_to_kelvin(a)
        ax.hist(vals, bins=60)
        ax.set_title(f"{b} (Kelvin)")
    else:
        ax.hist(a, bins=60)
        ax.set_title(b)
    ax.set_yscale("log")

plt.tight_layout()
plt.show()

# Per-example band stats across all files
rows = []
files = list_tfrecords(TFRECORD_DIR)
print("Files:", len(files))

for p in files:
    ds = tf.data.TFRecordDataset(p, compression_type="GZIP")
    for raw in ds:
        f = parse_example(raw.numpy())
        # ID info if present
        cc = f["country"].bytes_list.value[0].decode("utf-8") if "country" in f else None
        yr = int(f["year"].float_list.value[0]) if "year" in f else None
        cl = int(f["cluster_index"].float_list.value[0]) if "cluster_index" in f else None

        # Per-band moments
        band_stats = {}
        for b in BAND_ORDER:
            a = np.asarray(f[b].float_list.value, np.float32) if b in f else np.array([], np.float32)
            if b == "TEMP1" and a.size:
                a = temp_to_kelvin(a)
            if a.size == PIXELS_PER_IMAGE:
                band_stats[b+"_min"]  = float(np.nanmin(a))
                band_stats[b+"_max"]  = float(np.nanmax(a))
                band_stats[b+"_mean"] = float(np.nanmean(a))
                band_stats[b+"_std"]  = float(np.nanstd(a))
                band_stats[b+"_finite_frac"] = float(np.isfinite(a).mean())
            else:
                band_stats[b+"_min"] = band_stats[b+"_max"] = band_stats[b+"_mean"] = band_stats[b+"_std"] = np.nan
                band_stats[b+"_finite_frac"] = 0.0

        rows.append({
            "file": os.path.basename(p),
            "country": cc, "year": yr, "cluster": cl,
            **band_stats
        })

df = pd.DataFrame(rows)
out_csv = os.path.join(OUT_DIR, "band_stats_all.csv")
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# Dataset-level means/stds per band for reference
summary = {}
for b in BAND_ORDER:
    m = df[b+"_mean"].dropna()
    summary[b] = {"dataset_mean_of_means": float(m.mean()) if len(m) else np.nan,
                  "dataset_std_of_means":  float(m.std())  if len(m) else np.nan}
summary_csv = os.path.join(OUT_DIR, "dataset_means_like_constants.csv")
pd.DataFrame(summary).to_csv(summary_csv)
print("Saved:", summary_csv)
pd.DataFrame(summary)


In [ ]:
# "Bad pixel" = non-finite; for reflectance bands also count negatives; NL < 0.
BAD_REFLECTANCE = ['BLUE','GREEN','RED','NIR','SWIR1','SWIR2']

def count_bad_pixels(fmap):
    out = {}
    for b in BAND_ORDER:
        a = np.asarray(fmap[b].float_list.value, np.float32) if b in fmap else np.array([], np.float32)
        if b == "TEMP1" and a.size:
            a = temp_to_kelvin(a)  # convert only; no thresholding
        if a.size != PIXELS_PER_IMAGE:
            out[b] = PIXELS_PER_IMAGE  # treat missing as all bad
            continue
        bad = ~np.isfinite(a)
        if b in BAD_REFLECTANCE:
            bad |= (a < 0)
        if b == "NIGHTLIGHTS":
            bad |= (a < 0)
        out[b] = int(bad.sum())
    out["bad_total"] = int(sum(out[b] for b in BAND_ORDER))
    return out

wrows = []
files = list_tfrecords(TFRECORD_DIR)
for p in files:
    for raw in tf.data.TFRecordDataset(p, compression_type="GZIP"):
        f = parse_example(raw.numpy())
        cc = f["country"].bytes_list.value[0].decode("utf-8") if "country" in f else None
        yr = int(f["year"].float_list.value[0]) if "year" in f else None
        cl = int(f["cluster_index"].float_list.value[0]) if "cluster_index" in f else None
        counts = count_bad_pixels(f)
        wrows.append({"file": os.path.basename(p), "country": cc, "year": yr, "cluster": cl, **counts})

baddf = pd.DataFrame(wrows).sort_values("bad_total", ascending=False)
topk = baddf.head(12)
top_csv = os.path.join(OUT_DIR, "worst_bad_pixels.csv")
topk.to_csv(top_csv, index=False)
print("Saved:", top_csv)
topk.head(100)


In [ ]:
# Plot histograms for the same example picked in Cell 2
fig, axes = plt.subplots(2, 4, figsize=(14,6))
axes = axes.ravel()

for i, b in enumerate(BAND_ORDER):
    ax = axes[i]
    a = arrs[b]
    if a.size != PIXELS_PER_IMAGE:
        ax.set_title(f"{b} (missing)")
        ax.axis("off"); continue
    if b == "TEMP1":
        vals = temp_to_kelvin(a)
        ax.hist(vals, bins=60)
        ax.set_title(f"{b} (Kelvin)")
    else:
        ax.hist(a, bins=60)
        ax.set_title(b)
    ax.set_yscale("log")

plt.tight_layout()
plt.show()


In [ ]:
rows = []
files = list_tfrecords(TFRECORD_DIR)
print("Files:", len(files))

for p in files:
    ds = tf.data.TFRecordDataset(p, compression_type="GZIP")
    for raw in ds:
        f = parse_example(raw.numpy())
        # ID info if present
        cc = f["country"].bytes_list.value[0].decode("utf-8") if "country" in f else None
        yr = int(f["year"].float_list.value[0]) if "year" in f else None
        cl = int(f["cluster_index"].float_list.value[0]) if "cluster_index" in f else None

        # Per-band moments
        band_stats = {}
        for b in BAND_ORDER:
            a = np.asarray(f[b].float_list.value, np.float32) if b in f else np.array([], np.float32)
            if b == "TEMP1" and a.size:
                a = temp_to_kelvin(a)
            if a.size == PIXELS_PER_IMAGE:
                band_stats[b+"_min"]  = float(np.nanmin(a))
                band_stats[b+"_max"]  = float(np.nanmax(a))
                band_stats[b+"_mean"] = float(np.nanmean(a))
                band_stats[b+"_std"]  = float(np.nanstd(a))
                band_stats[b+"_finite_frac"] = float(np.isfinite(a).mean())
            else:
                band_stats[b+"_min"] = band_stats[b+"_max"] = band_stats[b+"_mean"] = band_stats[b+"_std"] = np.nan
                band_stats[b+"_finite_frac"] = 0.0

        rows.append({
            "file": os.path.basename(p),
            "country": cc, "year": yr, "cluster": cl,
            **band_stats
        })

df2 = pd.DataFrame(rows)
out_csv2 = os.path.join(OUT_DIR, "band_stats_all_dup.csv")
df2.to_csv(out_csv2, index=False)
print("Saved:", out_csv2)
